# CHASM: basic usage and custom statistics

This notebook demonstrates a single simulated Gaussian mean-change experiment,
the article metric selection, a classical-mean-only call, and a custom
Wasserstein divergence statistic.


In [1]:
import numpy as np

from chasm import (
    bootstraping_divergences,
    bootstraping_one_dim_stats,
    chasm_bounds_main,
    chasm_main,
)
from scipy.stats import wasserstein_distance


## Simulated Gaussian mean change

Both segments have identity covariance. The mean changes from `[0, 0]` to
`[1, 1]`, matching the `mean_change` scenario in
`experiments/configs/gaussian.json`.


In [2]:
np.random.seed(2024)
n_observations = 100
covariance = np.eye(2)

segment1 = np.random.multivariate_normal(
    mean=[0.0, 0.0],
    cov=covariance,
    size=n_observations,
)
segment2 = np.random.multivariate_normal(
    mean=[1.0, 1.0],
    cov=covariance,
    size=n_observations,
)


In [3]:
all_results = chasm_main(
    segment1,
    segment2,
    confidence_level=0.95,
    n_resamples=50,
)

for metric_name, metric_result in all_results.items():
    print(metric_name, metric_result)


sig_l_mean {'decision': 1, 'upper_bound': None, 'lower_bound': 2.2474472156350043, 'stat_value': 1.9288055169568619, 'imf': 0.14177939150758292}
sig_l_scale {'decision': 0, 'upper_bound': None, 'lower_bound': 1.7693058517120819, 'stat_value': 2.761712787785507, 'imf': 0.0}
sig_l_skewness {'decision': 0, 'upper_bound': None, 'lower_bound': 1.872070453809618, 'stat_value': 3.433900394374175, 'imf': 0.0}
sig_l_kurtosis {'decision': 0, 'upper_bound': None, 'lower_bound': 1.9072627390650378, 'stat_value': 2.3448794569550837, 'imf': 0.0}
sig_robust_mean {'decision': 1, 'upper_bound': None, 'lower_bound': 0.7260362825068198, 'stat_value': 0.21749797263010695, 'imf': 0.700430986893463}
sig_robust_scale {'decision': 0, 'upper_bound': None, 'lower_bound': 0.28828265893694827, 'stat_value': 1.4368894047832277, 'imf': 0.0}
sig_exact_skewness {'decision': 0, 'upper_bound': None, 'lower_bound': 0.354509642516262, 'stat_value': 2.068813280750232, 'imf': 0.0}
sig_exact_kurtosis {'decision': 0, 'upper_

## Metrics used in the article

`chasm_bounds_main` selects the article metrics by default. A different list
can be passed through its `metrics` argument.


In [4]:
article_results = chasm_bounds_main(
    segment1,
    segment2,
    confidence_level=0.95,
    n_resamples=50,
)
print(list(article_results))

selected_results = chasm_bounds_main(
    segment1,
    segment2,
    metrics=("sig_robust_mean", "sig_kl_divergence"),
    n_resamples=50,
)
print(selected_results)


['sig_l_mean', 'sig_l_scale', 'sig_l_skewness', 'sig_l_kurtosis', 'sig_robust_scale', 'sig_spearman_corr', 'sig_bhattacharyya_dist', 'sig_jeffrey_dist']
{'sig_robust_mean': {'decision': 1, 'upper_bound': None, 'lower_bound': 0.6909863318415559, 'stat_value': 0.21749797263010695, 'imf': 0.6852354922123416}, 'sig_kl_divergence': {'decision': 0, 'upper_bound': 4.610700789529181, 'lower_bound': None, 'stat_value': 3.946845693362066, 'imf': 0.0}}


## Classical mean only

A one-dimensional custom statistic should return a lower-is-more-significant
score when used with `bootstraping_one_dim_stats`. The result is a dictionary
with the same keys as built-in metrics.


In [5]:
def classical_mean_score(
    first: np.ndarray,
    second: np.ndarray,
    axis: int | None = None,
) -> np.ndarray:
    """Return a lower-is-more-significant score based on mean difference."""
    difference = np.abs(
        np.mean(first, axis=axis) - np.mean(second, axis=axis)
    )
    return np.array([-np.log(difference + np.finfo(float).eps)])


mean_result = bootstraping_one_dim_stats(
    segment1,
    segment2,
    statistic=classical_mean_score,
    confidence_level=0.95,
    n_resamples=50,
)
print(mean_result)


{'decision': 1, 'upper_bound': None, 'lower_bound': 0.9618945952992913, 'stat_value': 0.16337487611742357, 'imf': 0.8301530366052324}


## Custom distribution divergence

A divergence-like statistic is compared with an upper confidence bound. The
callable receives the two sample arrays and may aggregate per-column distances
into one scalar.


In [6]:
def wasserstein_statistic(
    first: np.ndarray,
    second: np.ndarray,
    axis: int | None = None,
) -> np.ndarray:
    """Return the maximum per-column Wasserstein distance."""
    del axis
    distances = [
        wasserstein_distance(first[:, column], second[:, column])
        for column in range(first.shape[1])
    ]
    return np.array([max(distances)])


divergence_result = bootstraping_divergences(
    segment1,
    segment2,
    statistic=wasserstein_statistic,
    confidence_level=0.95,
    n_resamples=50,
)
print(divergence_result)


{'decision': 1, 'upper_bound': 0.5407372908315676, 'lower_bound': None, 'stat_value': 0.8492727566584387, 'imf': 0.5705829264195794}
